# Tox21 GNN — Exploratory Analysis & Results

This notebook covers:
1. Dataset EDA
2. Molecule visualization
3. Training results
4. Model comparison

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml, json, torch

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

TOX21_TASKS = [
    'NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase',
    'NR-ER','NR-ER-LBD','NR-PPAR-gamma',
    'SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53'
]

## 1. Dataset EDA

In [ ]:
train_df = pd.read_csv('../data/tox21/train.csv')
val_df   = pd.read_csv('../data/tox21/val.csv')
test_df  = pd.read_csv('../data/tox21/test.csv')

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
train_df[TOX21_TASKS].describe()

In [ ]:
# Class balance heatmap
pos_rates = {}
for split_name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    pos_rates[split_name] = [
        df[t].dropna().mean() for t in TOX21_TASKS
    ]

pr_df = pd.DataFrame(pos_rates, index=TOX21_TASKS)

fig, ax = plt.subplots(figsize=(6, 7))
sns.heatmap(pr_df, annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=0, vmax=0.5, ax=ax, linewidths=0.5)
ax.set_title('Positive Rate per Task & Split', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Missing value rates
missing = train_df[TOX21_TASKS].isna().mean() * 100
fig, ax = plt.subplots(figsize=(8, 4))
missing.sort_values().plot.barh(ax=ax, color='#4C72B0', edgecolor='white')
ax.set_xlabel('Missing Rate (%)')
ax.set_title('Missing Label Rate (Train Set)', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Molecule Featurization

In [ ]:
from data.dataset import smiles_to_graph, ATOM_FEATURE_DIM, BOND_FEATURE_DIM

print(f'Atom feature dim : {ATOM_FEATURE_DIM}')
print(f'Bond feature dim : {BOND_FEATURE_DIM}')

examples = {
    'Aspirin'  : 'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine' : 'Cn1c(=O)c2c(ncn2C)n(c1=O)C',
    'Benzene'  : 'c1ccccc1',
    'Ethanol'  : 'CCO',
}
for name, smi in examples.items():
    g = smiles_to_graph(smi)
    print(f'{name:10s}: {g.num_nodes} atoms, {g.num_edges} edges')

## 3. Training Results

In [ ]:
from utils.visualize import plot_training_curves, plot_task_aucs, plot_roc_curves

# Load saved results
import os, glob
result_files = glob.glob('../results/test_results_*.json')
all_results = {}
for f in result_files:
    with open(f) as fp:
        r = json.load(fp)
    all_results[r['model']] = r
    print(f"{r['model']:5s}: Mean AUC={r['mean_auc']:.4f}")

In [ ]:
# Compare models
if all_results:
    models = list(all_results.keys())
    task_aucs = {m: [all_results[m]['per_task'][t]['auc'] for t in TOX21_TASKS] for m in models}
    
    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(TOX21_TASKS))
    width = 0.25
    
    colors = ['#4C72B0', '#DD8452', '#55A868']
    for i, (m, color) in enumerate(zip(models, colors)):
        aucs = [v if not np.isnan(v) else 0 for v in task_aucs[m]]
        ax.bar(x + i*width, aucs, width, label=m, color=color, edgecolor='white')
    
    ax.set_xticks(x + width)
    ax.set_xticklabels(TOX21_TASKS, rotation=45, ha='right')
    ax.set_ylabel('ROC-AUC')
    ax.set_ylim(0, 1)
    ax.set_title('Model Comparison — Per-Task ROC-AUC', fontweight='bold')
    ax.legend()
    ax.axhline(0.5, color='grey', linestyle=':', linewidth=0.8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

## 4. Single Molecule Prediction

In [ ]:
from scripts.predict import predict_smiles
from data.dataset import ATOM_FEATURE_DIM
from models.gnn import build_model

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cpu')
model = build_model(cfg, in_dim=ATOM_FEATURE_DIM)
ckpt = torch.load(f"../results/checkpoints/best_{cfg['model']['type'].upper()}.pt", map_location='cpu')
model.load_state_dict(ckpt['state_dict'])

smiles_to_test = [
    'CC(=O)Oc1ccccc1C(=O)O',   # Aspirin
    'c1ccc2cc3ccccc3cc2c1',     # Anthracene (PAH — likely toxic)
    'CCO',                       # Ethanol
    'ClC(Cl)(Cl)Cl',             # Carbon tetrachloride (hepatotoxic)
]

results = predict_smiles(smiles_to_test, model, device)
for r in results:
    toxic = r['toxic_tasks']
    print(f"{r['smiles'][:40]:<42}  "
          f"{'TOXIC' if toxic else 'SAFE':6s}  {', '.join(toxic) if toxic else '—'}")